# Choice-logit weights across training: structured, then drifting problems

Samborska et al. (2022, *Nat Neurosci*, Fig. 1i-l) fitted a logistic regression of choices on past
choices and past outcome x choice in every problem of a two-port reversal task and found that the
weights grow across problems: the mice integrate more of the recent history as they learn the task
structure. Here the standard three-arm choice logit (reward magnitude and choice at lags 1-5 per
port, a bias per tower) is fitted to each mouse's problems one at a time, through the structured
(blocked) problems that come first and then the drifting ones
(`src/behavior_analysis/choice_logit/logit_by_problem.py`).


In [ ]:
# ruff: noqa: E402  (the repository root is put on sys.path first)
import sys
from pathlib import Path

from IPython.display import Image, display

REPO_ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.behavior_visualization.choice_logit import plot_logit_by_problem as plots
from src.pipelines.choice_logit import logit_by_problem as pipe


In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
# The last drifting problem kept per cohort (None: all); as the rank-signature notebook.
MAX_DRIFTING_BY_COHORT = {1: 7, 3: None}
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "logit-by-problem"


## Load and fit

Structured problems are the blocked problems before a mouse's first drifting problem; blocked
problems that come later (cohort 1, after surgery) and drifting problems after the cutoff are left
out. Problems with fewer than 150 trials get no fit.


In [ ]:
trials = pipe.load_training_trial_table(COHORTS, DATA_ROOT)
results = pipe.run_logit_by_problem(trials, MAX_DRIFTING_BY_COHORT, OUTPUT_DIR)
fitted = results["magnitude"].weights.groupby(["phase", "phase_problem"])["mouse"].nunique()
display(fitted.rename("mice"))


## Weights by problem

Lag-1 weight and the mean of lags 2-5 (as the paper's Fig. 1k,l), in two codings of the history:

- **Magnitude** (the standard logit): per port, choice (1 if chosen) and reward (the pulses received
  there). The reward weight is the analogue of the paper's outcome x choice term, per pulse; the
  choice weight is the pull of a past choice that paid 0.
- **Outcome x choice** (the paper's coding): choice, and outcome x choice (+1 if the choice paid
  anything, -1 if it paid 0). The choice weight is the average pull of a past choice, rewarded or
  not; the outcome x choice weight is half the difference between rewarded and unrewarded
  (win-stay versus lose-shift). The magnitude is ignored, as the paper's reward was binary.

The caption file beside each figure has the slope and step tests.


In [ ]:
for coding, result in results.items():
    name = plots.figure_name(plots.WEIGHTS_BY_PROBLEM, coding)
    display(Image(filename=str(OUTPUT_DIR / f"{name}.png")))
    display(result.tests.round(4))


## Kernels by problem

Every problem's history and choice kernels (lags 1-5) in turn, as the paper's Fig. 1i,j, in both
codings.


In [ ]:
for coding in results:
    name = plots.figure_name(plots.KERNELS_BY_PROBLEM, coding)
    display(Image(filename=str(OUTPUT_DIR / f"{name}.png")))
